# 📘 Annotated edition · 04_sql_transformations

A copy of `notebooks/04_sql_transformations.ipynb`. Original cells and saved outputs are unchanged. New **📘 Explanation** and **✅ Answer** cells are added.

**The point of this notebook:** load the clean tables into a database (SQLite by default, PostgreSQL if `KALPA_PG` is set) and recompute the key numbers **in SQL**. SQL is what finance can re-run without Python, and two independent implementations agreeing is strong evidence that a number is right.

**What the saved run shows, including a disagreement worth knowing about:**
- the SQL tree gives **433,611** tests in H1 2025 and **455,383** in H1 2026 (+5.02%);
- pandas in notebook 05 gives **431,261** and **452,933** (+5.03%) for the "same" periods;
- the ~2,400-test gap is **30 June**: pandas' `between("2025-01-01", "2025-06-30")` stops at **00:00 on 30 June**, while SQL's `slot_at < '2025-07-01'` includes the whole day. **SQL is right.** Details in the ✅ answers.

> ℹ️ **One line changed in this annotated copy:** `ROOT = ... if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else ...`. The original only recognised a folder called `notebooks`. Run from `notebook_new/`, it would have looked for `notebook_new/data/raw` and failed. Nothing else in the code was changed.

# 04 · The same questions in SQL, at warehouse scale

**Produces:** the clean tables loaded into a database, and the outputs of `sql/03_profile.sql`, `sql/04_transformations.sql`, `sql/05_performance.sql`.

**Needs:** notebook 03 run; either a PostgreSQL connection or nothing (the notebook falls back to SQLite from the standard library).

Why SQL after pandas: the warehouse is where finance will re-run your numbers, the logical order of a query (FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY) is the interview question that never goes away, and window functions answer "compared with the previous month, per city" in one pass where pandas needs three.

#### 📘 Explanation: why SQL after pandas

1. **The warehouse is where numbers get re-run.** Finance won't run your notebook, but they will run a SQL query.
2. **Logical query order** (FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY) is a classic interview question, answered below.
3. **Window functions** compute "this month vs the previous month, per city" in **one pass** while keeping every row. In pandas that's a groupby, a `shift`, and a merge back.

## CONFIG

#### 📘 Explanation: the CONFIG cell, line by line

This cell is the same in notebooks 01–06. Explained once in full:

| Code | What it does and why |
|---|---|
| `import sys, pathlib, time` | `sys` to edit the import path, `pathlib` for file paths that work on Windows and Linux, `time` to time steps. |
| `ROOT = pathlib.Path.cwd().parent if ... == "notebooks" else pathlib.Path.cwd()` | Finds the **project root**. Jupyter usually starts inside the notebook's folder, so the root is one level up. When run from the root itself (for example a script), `cwd()` is already the root. Every path below is built from `ROOT`, so the notebook works on any machine. |
| `sys.path.insert(0, str(ROOT / "src"))` | Puts `src/` at the front of Python's import search path, so `from kalpa import ...` finds the project's own package without `pip install`. |
| `import numpy as np` / `import pandas as pd` | The two core libraries: arrays and fast maths / tables. |
| `from kalpa import loaders, models, checks, logs` | The shared project package: **loaders** read the raw files, **models** are pydantic "contracts" for a valid record, **checks** print PASS/FAIL, **logs** append rows to the markdown logs in `docs/`. |
| `pd.set_option("display.max_columns", 40)` / `("display.width", 160)` | Print wide tables in full instead of truncating them with `...`. |
| `CONFIG = {...}` | All run-specific settings in one place (see notebook 00): brief, group, `raw` and `clean` folders as `pathlib` paths, the two half-years being compared, the seed, and `workers` (`None` means use every CPU core in the process pool). |
| `CONFIG["clean"].mkdir(parents=True, exist_ok=True)` | Creates `data/clean/` if it doesn't exist, and doesn't fail if it does. A fresh clone can run straight away. |
| `rng = np.random.default_rng(CONFIG["seed"])` | A seeded random generator **object**. Anything random (the shuffle test in 05) uses `rng`, so results are reproducible. |
| `print(pd.__version__, np.__version__)` | Prints library versions (**2.3.3** and **2.3.5** in the saved run). If a re-run elsewhere gives different numbers, a version difference is the first suspect. pandas 3.x, for example, changed the default string dtype. |


**Extra lines in this notebook:**
- `import sqlite3, os`: SQLite ships with Python, so the notebook needs no database server.
- `PG_DSN = os.environ.get("KALPA_PG")`: if the environment variable `KALPA_PG` holds a PostgreSQL connection string, PostgreSQL is used. Otherwise it's `None` and SQLite is used.
- `DB_PATH = CONFIG["clean"] / "kalpa.sqlite"`: the SQLite database is a single file in `data/clean/`.
- `SQL_DIR = ROOT / "sql"`: where the `.sql` files live.

In [1]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "<LETTER>",
    "group": "<N>",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

import sqlite3, os
PG_DSN = os.environ.get("KALPA_PG")          # e.g. "host=localhost dbname=kalpa user=postgres"; leave unset for SQLite
DB_PATH = CONFIG["clean"] / "kalpa.sqlite"
SQL_DIR = ROOT / "sql"

2.3.3 2.3.5


## 1 · DO: load the clean tables

#### 📘 Explanation: loading the clean tables into the database

- `tables = {...}` reads each clean CSV with the right date columns parsed:
  - `bookings` (563,380 rows, with collection status and person_id), `patients`, `invoices`, `payments`;
  - `plan`: the raw plan file is **wide** (one column per month, `2026-01` ... `2026-06`). `.melt(id_vars=["city","metric"], var_name="month", value_name="target_tests")` turns it **long**, one row per city per month, so it can be joined to actuals on `(city, month)`. 6 cities × 6 months = 36 rows.
- `tables["plan"]["city"].replace({"Bangalore": "Bengaluru"})`: the plan uses the old spelling too. Without this, Bengaluru's actuals would never join to its plan.
- If `PG_DSN` is set, it connects to PostgreSQL with `psycopg2`. Otherwise it **deletes any old SQLite file** (`DB_PATH.unlink()`) so every run starts clean (*idempotent*: running twice gives the same database), then connects.
- `df.to_sql(name, con, if_exists="replace", index=False)` creates (or replaces) a table and inserts the rows. Note: on PostgreSQL, `to_sql` officially needs a SQLAlchemy engine, so with a raw psycopg2 connection use `sql/02_load.sql` (`\copy`) instead.

In [2]:
tables = {
    "bookings": pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["created_at", "slot_at", "collected_at"], dtype={"rescheduled_from": "string", "status_at_export": "string"}),
    "patients": pd.read_csv(CONFIG["clean"] / "patients_clean.csv.gz", dtype="string"),
    "invoices": pd.read_csv(CONFIG["clean"] / "invoices_clean.csv.gz", parse_dates=["invoice_date"]),
    "payments": pd.read_csv(CONFIG["clean"] / "payments_clean.csv.gz", parse_dates=["created_at"]),
    "plan": pd.read_csv(CONFIG["raw"] / "plan" / "plan_2026_H1.csv").melt(id_vars=["city", "metric"], var_name="month", value_name="target_tests"),
}
tables["plan"]["city"] = tables["plan"]["city"].replace({"Bangalore": "Bengaluru"})
if PG_DSN:
    import psycopg2
    con = psycopg2.connect(PG_DSN)
    print("PostgreSQL:", con.server_version)
else:
    if DB_PATH.exists(): DB_PATH.unlink()
    con = sqlite3.connect(DB_PATH)
    print("SQLite at", DB_PATH)
for name, df in tables.items():
    df.to_sql(name, con, if_exists="replace", index=False)
    print(f"{name}: {len(df):,} rows")

SQLite at c:\Users\Lenovo\Desktop\Pure_Kalpa_Data_Folder\data\clean\kalpa.sqlite
bookings: 563,380 rows
patients: 303,153 rows
invoices: 470,776 rows
payments: 473,507 rows
plan: 36 rows


#### 📘 Reading the output
SQLite file created in `data/clean/`. Row counts match the clean CSVs exactly: bookings **563,380**, patients **303,153**, invoices **470,776**, payments **473,507**, plan **36**. A load that lost rows would show here.

## 2 · DO: run the SQL files

The files in `sql/` are written for PostgreSQL. `03_profile.sql` and `04_transformations.sql` also run on SQLite with the small substitutions `run_sql` makes; `05_performance.sql` (indexes and EXPLAIN) is PostgreSQL only and is skipped on SQLite.

#### 📘 Explanation
The `.sql` files are written in **PostgreSQL** dialect. `run_sql` makes a few text substitutions so the same files run on SQLite. Performance work (indexes, `EXPLAIN ANALYZE`) only means something on the real warehouse engine, so it's skipped on SQLite.

#### 📘 Explanation: `run_sql`, a tiny SQL runner

1. `text = pathlib.Path(path).read_text()` reads the whole `.sql` file.
2. `text.split(";\n")` splits it into statements at semicolons that end a line.
3. For each statement:
   - `title`: the first `--` comment line, used as a heading in the output.
   - `stmt`: the statement with comment lines and psql meta-commands (lines starting with `\`) removed.
4. **SQLite translation** (string replacements):
   - `date_trunc('month', x)` → `strftime('%Y-%m', x)` (both give "the month of x"),
   - `::date` and `::numeric` (PostgreSQL casts) are removed,
   - `EXTRACT(YEAR FROM x)` → `strftime('%Y', x)`.
   - statements with `EXPLAIN`, `CREATE INDEX` or starting with `ANALYZE` are skipped.
5. `pd.read_sql_query(stmt, con)` runs the statement and returns a DataFrame.
6. Any error is caught and turned into a one-row DataFrame with the error message, so one bad query doesn't stop the rest. (The trade-off: you have to *read* the output to notice an error.)

Then `03_profile.sql` runs and each result prints under its title. **Why string comparison of dates works in SQLite:** pandas writes timestamps as text `'2025-06-30 14:05:00'`. ISO format sorts lexicographically in time order, so `slot_at < '2025-07-01'` is a correct date comparison.

In [3]:
def run_sql(path, con):
    text = pathlib.Path(path).read_text()
    is_sqlite = isinstance(con, sqlite3.Connection)
    outputs = []
    for raw in text.split(";\n"):
        lines = [ln for ln in raw.strip().splitlines() if ln.strip()]
        title = next((ln.lstrip("- ").strip() for ln in lines if ln.strip().startswith("--")), "")[:90]
        stmt = "\n".join(ln for ln in lines if not ln.strip().startswith("--") and not ln.strip().startswith("\\"))
        if not stmt.strip():
            continue
        if is_sqlite:
            stmt = (stmt.replace("date_trunc('month', ", "strftime('%Y-%m', ").replace("::date", "").replace("::numeric", "")
                        .replace("EXTRACT(YEAR FROM ", "strftime('%Y', "))
            if "EXPLAIN" in stmt.upper() or "CREATE INDEX" in stmt.upper() or stmt.upper().startswith("ANALYZE"):
                continue
        try:
            outputs.append((title or stmt.splitlines()[0][:70], pd.read_sql_query(stmt, con)))
        except Exception as err:
            outputs.append((title or stmt.splitlines()[0][:70], pd.DataFrame({"error": [str(err)[:200]]})))
    return outputs

for title, df in run_sql(SQL_DIR / "03_profile.sql", con):
    print("\n==", title); print(df.head(12).to_string(index=False))


== Profile in SQL: the same questions as notebook 01, asked of the loaded tables.
     tbl  rows_  distinct_keys          first_slot           last_slot
bookings 563380         563380 2025-01-01 06:00:00 2026-07-04 10:50:00

== Null rates per column, the long way: one line per column so the reader sees the shape of t
         col  null_rate
collected_at     0.1735
   person_id     0.0000
      status     0.0000

== Grain check: rows per booking must be 1 here; if it is not, the load doubled something
 max_rows_per_booking
                    1

== Bookings per system per month: the migration is visible as a change of system, not of volu
system   month  bookings  tests
legacy 2025-01     31902  73094
legacy 2025-02     31528  72551
legacy 2025-03     32142  73958
legacy 2025-04     31296  71974
legacy 2025-05     31166  71153
legacy 2025-06     30769  70881
legacy 2025-07     30687  70659
legacy 2025-08     29501  67622
legacy 2025-09     31129  71346
legacy 2025-10     30098  69348
le

#### 📘 Reading the output (`03_profile.sql`)

- **Table profile:** bookings 563,380 rows = 563,380 distinct booking ids (one row per booking). Slots from 2025-01-01 06:00 to **2026-07-04 10:50** (the July nova slots).
- **Null rates:** `collected_at` **17.35%**. That's 18,269 unmatched + 78,548 no-shows (a no-show has no collection time) + 936 blanked clock-skew timestamps ≈ 97,750 of 563,380. `person_id` and `status` 0%.
- **Grain check:** `max_rows_per_booking = 1`, so the load didn't duplicate anything.
- **Bookings per system per month:** the migration shows up as a switch from legacy to nova for Chennai and Jaipur in April 2026, **not** as a change in volume.
- **Payment references well-formed after normalisation:** the same 100% as pandas.

#### 📘 Explanation: `04_transformations.sql`, the methods as CTE pipelines

Each query is a chain of **CTEs** (`WITH name AS (...)`), named steps you read top to bottom like a recipe:
1. **The tree** (`halves` → `agg`): labels each booking `H1_2025`/`H1_2026` with **half-open** intervals (`>= '2025-01-01' AND < '2025-07-01'`), excludes superseded records (`status_at_export IS DISTINCT FROM 'rescheduled'`, which, unlike `<>`, treats NULL as "distinct" and so **keeps legacy rows**: the SQL version of the `pd.NA` trap), then computes tests, visits, people, visits per person and tests per visit.
2. **Monthly rollup with window functions** (see the interview answer below): `LAG` for the previous month, the month-on-month %, a running total and the rank among cities in each month.
3. **Actual against plan**: plan `LEFT JOIN` monthly actuals on `(city, month)`, with `coalesce(m.tests, 0)` so a month with no bookings shows 0 rather than NULL.
4. **The bridge**: Q2 2026 invoices classified as *never paid / paid after cutoff / collected in quarter*.
5. **Anti-join**: captured payments whose invoice doesn't exist (`LEFT JOIN ... WHERE i.invoice_id IS NULL`). This is the list finance acts on.
6. **Retries**: invoices with more than one payment row (`HAVING count(*) > 1`), split into captured and failed.
7. **No-show rate per clinic**, with the bookings denominator beside it (only collected + no-show bookings).

In [4]:
for title, df in run_sql(SQL_DIR / "04_transformations.sql", con):
    print("\n==", title); print(df.head(12).to_string(index=False))


== The methods, as CTE pipelines a finance analyst can read top to bottom.
   half  tests  visits  people  visits_per_person  tests_per_visit
H1_2025 433611  188803   94275             2.0027           2.2966
H1_2026 455383  194020   97827             1.9833           2.3471

== 2. Monthly rollup with window functions. Grain: one row per city per month.
     city   month  tests  prev_month  mom_pct  running_total  rank_in_month
Ahmedabad 2025-01  12728         NaN      NaN          12728              3
Ahmedabad 2025-02  12476     12728.0     -2.0          25204              3
Ahmedabad 2025-03  12462     12476.0     -0.1          37666              3
Ahmedabad 2025-04  13160     12462.0      5.6          50826              3
Ahmedabad 2025-05  12659     13160.0     -3.8          63485              3
Ahmedabad 2025-06  12995     12659.0      2.7          76480              3
Ahmedabad 2025-07  12368     12995.0     -4.8          88848              3
Ahmedabad 2025-08  11724     12368.

#### 📘 Reading the output

**1. Tree:** H1 2025: 433,611 tests, 188,803 visits, 94,275 people (2.0027 visits per person, 2.2966 tests per visit). H1 2026: 455,383 tests, 194,020 visits, 97,827 people (1.9833 and 2.3471). Growth: tests **+5.02%**, people +3.77%, visits per person −0.97%, tests per visit +2.20%.

**2. Monthly rollup:** for example Ahmedabad 2025-01: 12,728 tests, no previous month (NULL, the first month), running total 12,728, rank 3 of 6 cities that month. 2025-02: 12,476 (−2.0%), running total 25,204.

**3. Actual vs plan (2026 H1, summed per city):**

| City | Plan | Actual | Gap |
|---|---|---|---|
| Ahmedabad | 90,773 | 84,523 | −6.9% |
| Bengaluru | 120,085 | 106,973 | −10.9% |
| Chennai | 93,912 | 80,502 | −14.3% |
| Hyderabad | 86,421 | 78,131 | −9.6% |
| Jaipur | 52,372 | 40,113 | **−23.4%** |
| Pune | 72,177 | 65,141 | −9.7% |
| **Total** | **515,740** | **455,383** | **−11.7%** |

The plan assumed about **+18.9%** over H1 2025 actuals (433,611). Actual growth was +5.0%, so **every city missed plan**, Jaipur by the most.

(Queries 4–7 print the bridge bars, the top-20 orphan payments, the top-20 retry invoices, and clinic no-show rates. Notebook 05 explains each number.)

> **Interview angle.** Write the logical order in which a SELECT with a GROUP BY and a HAVING is evaluated, and use it to explain why a column alias from SELECT cannot be used in WHERE.
>
> *What a good answer covers:* FROM and JOIN, then WHERE, then GROUP BY, then HAVING, then SELECT, then ORDER BY and LIMIT. WHERE runs before SELECT has produced the alias, so the alias does not exist yet; HAVING runs after grouping so it can see aggregates; ORDER BY runs after SELECT so it can see aliases.

#### ✅ Answer: the logical order of a SELECT with GROUP BY and HAVING, and why a SELECT alias can't be used in WHERE

The **written** order is `SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY ... LIMIT`. The **logical evaluation** order is:

| Step | Clause | What exists at this point |
|---|---|---|
| 1 | **FROM** / **JOIN** (and ON) | the raw rows of the joined tables |
| 2 | **WHERE** | filters **individual rows**. Only table columns exist. No aggregates, no aliases. |
| 3 | **GROUP BY** | collapses rows into groups |
| 4 | **HAVING** | filters **groups**, so it can use aggregates such as `count(*) > 1` |
| 5 | **SELECT** | computes output expressions and **creates aliases**. Window functions are evaluated here, after grouping. |
| 6 | **DISTINCT** | removes duplicate output rows |
| 7 | **ORDER BY** | sorts, and **can** use aliases, because SELECT has run |
| 8 | **LIMIT / OFFSET** | cuts the sorted result |

**Why an alias can't be used in WHERE:** in `SELECT amount_paise/100 AS amount_rs FROM payments WHERE amount_rs > 1000`, WHERE (step 2) runs **before** SELECT (step 5) has created `amount_rs`, so the name doesn't exist yet and you get "column amount_rs does not exist". Fixes: repeat the expression (`WHERE amount_paise/100 > 1000`), or compute it in a CTE or subquery first, then filter it in the outer query (which is why the transformations file is written as CTEs).

**Consequences of the same order:**
- You can't use an aggregate in WHERE (`WHERE count(*) > 1` fails). Use HAVING.
- You can't filter on a window function in WHERE or HAVING, because windows run in SELECT. Wrap it in a CTE: `WITH r AS (SELECT ..., RANK() OVER (...) AS rk ...) SELECT * FROM r WHERE rk = 1`.
- Some engines (SQLite, MySQL, and PostgreSQL in GROUP BY) accept aliases where the standard doesn't, but don't rely on it in code finance will port.

> **Interview angle.** When does a window function beat a GROUP BY, and what defines its result?
>
> *What a good answer covers:* When you need every row kept and a value computed over its neighbours: rank within a city, the previous month's value, a running total. The partition and the order inside the window define everything; a running total is only deterministic when the order has a tiebreaker.

#### ✅ Answer: when does a window function beat a GROUP BY, and what defines its result?

**GROUP BY collapses** rows: one output row per group. **A window function keeps every row** and adds a value computed over a set of related rows (the "window"). Use a window when you need both the detail and an aggregate over neighbours:
- **previous or next value**: `LAG(tests) OVER (PARTITION BY city ORDER BY month)` gives last month's tests on the same row, so month-on-month change is one expression;
- **rank within a group**: `RANK() OVER (PARTITION BY month ORDER BY tests DESC)`, which city was 1st this month;
- **running totals and moving averages**: `SUM(tests) OVER (PARTITION BY city ORDER BY month ROWS UNBOUNDED PRECEDING)`;
- **share of total**: `tests / SUM(tests) OVER (PARTITION BY month)`.

With GROUP BY you'd aggregate, then self-join the result to itself on `month - 1` (or `shift()` in pandas) and merge back. That's more code and an extra join that can fan out.

**What defines the result:**
1. **`PARTITION BY`**: which rows belong together (each city separately). Without it, the window is the whole table.
2. **`ORDER BY`** inside `OVER`: the sequence that LAG, RANK and running totals follow.
3. **The frame** (`ROWS BETWEEN ... AND ...`): which rows around the current one are included. With ORDER BY and no explicit frame, the default is `RANGE UNBOUNDED PRECEDING`, which includes **all rows tied** with the current one. That's a classic surprise.

**Determinism:** if the ORDER BY has ties, LAG and running totals can differ between runs or engines. Add a tiebreaker (`ORDER BY month, booking_id`). In query 2 it's deterministic because `month` is unique within each city, as the SQL comment notes.

## 3 · DO: performance, measured not guessed (PostgreSQL only)

#### 📘 Explanation
On PostgreSQL this runs `05_performance.sql`: `EXPLAIN ANALYZE` on a monthly rollup, then creates indexes on `bookings(city, slot_at)`, `payments(invoice_id)`, `invoices(invoice_date)`, refreshes statistics with `ANALYZE`, and re-runs `EXPLAIN ANALYZE` to compare plans and timings. On SQLite it just prints that it was skipped.

In [5]:
if PG_DSN:
    for title, df in run_sql(SQL_DIR / "05_performance.sql", con):
        print("\n==", title); print(df.head(20).to_string(index=False))
else:
    print("Skipped on SQLite. On PostgreSQL this file times a monthly rollup before and after an index on (city, slot_at) and prints EXPLAIN ANALYZE for both.")

Skipped on SQLite. On PostgreSQL this file times a monthly rollup before and after an index on (city, slot_at) and prints EXPLAIN ANALYZE for both.


#### 📘 Reading the output
Skipped, because the run used SQLite (no `KALPA_PG` set). To see the real plans, set `KALPA_PG` to a PostgreSQL connection string and load the tables with `sql/02_load.sql`.

> **Interview angle.** A query over 1.3 million booking rows that groups by city and month is slow. What do you look at first, and what would an index on (city, slot_at) change?
>
> *What a good answer covers:* EXPLAIN ANALYZE to see whether it is a sequential scan, a sort, or a hash aggregate that is slow, and whether the estimate is far from the actual rows. An index on (city, slot_at) lets the planner read rows in group order or skip to a date range, replacing a full scan plus sort with an index scan; it costs write time and space, so you add it for the queries finance will run weekly, not for a one-off.

#### ✅ Answer: a query over 1.3 M booking rows that groups by city and month is slow. What do you look at first, and what would an index on `(city, slot_at)` change?

**First: `EXPLAIN ANALYZE`.** It runs the query and shows the plan the database chose, with **actual** times and row counts per step. Look at:
1. **Which node takes the time**: a *Seq Scan* (reading the whole table), a *Sort* (especially one that spills to disk: "external merge Disk"), or a *HashAggregate* / *GroupAggregate*.
2. **Estimated vs actual rows.** A large mismatch (estimate 1,000, actual 1,300,000) means stale statistics, and the planner picks a bad strategy. Run `ANALYZE bookings`.
3. Whether the query **filters** at all. A rollup over *all* rows will always read all rows, and no index can avoid that.
4. **Computed grouping keys** such as `date_trunc('month', slot_at)`, which prevent a plain index on `slot_at` from being used for ordering.

**What an index on `(city, slot_at)` changes:**
- With a **date filter** (`WHERE slot_at >= '2026-01-01' AND slot_at < '2026-07-01'`), and better still a city filter, the planner can use an **index scan / bitmap scan** to read only the matching rows instead of the whole table.
- Rows can be read **already ordered by city, then time**, so a GroupAggregate can stream groups **without a separate sort**.
- If the index also covers the summed column (`INCLUDE (n_tests)` in PostgreSQL), an **index-only scan** avoids touching the table at all.
- For a full-table rollup with no WHERE, a sequential scan plus hash aggregate may *still* be fastest. That's why you measure before and after rather than assume.

**The costs:** every INSERT and UPDATE must also update the index (slower loads), it uses disk space, and it adds a choice for the planner. So add indexes for queries that **run repeatedly** (finance's weekly report), not for a one-off analysis.

## 4 · SUM

The same numbers you computed in pandas now come out of the database. Where they differ, one of the two is wrong; find out which before you go further. Write the SQL that finance will re-run into `sql/06_<your_brief>.sql` and reference it from the README.

## Before you move on

- Which of your pandas numbers did you reproduce in SQL, and did they agree to the rupee?
- Which query would you hand to finance, and could they read it without you in the room?
- What is the one index you would add if this ran weekly, and why not three?

#### ✅ Answers: "Before you move on"

**1. Which pandas numbers did you reproduce in SQL, and did they agree to the rupee?**

| Number | pandas | SQL | Agree? |
|---|---|---|---|
| bookings loaded | 563,380 | 563,380 | ✔ |
| one row per booking | `is_unique` = True | `max_rows_per_booking` = 1 | ✔ |
| invoices / payments rows | 470,776 / 473,507 | 470,776 / 473,507 | ✔ |
| payment refs well-formed after normalisation | 100% | 100% | ✔ |
| **H1 tests** | **431,261 / 452,933** (notebook 05) | **433,611 / 455,383** | ✘ **No** |
| H1 visits | 187,771 / 193,005 | 188,803 / 194,020 | ✘ |
| H1 people | 93,926 / 97,501 | 94,275 / 97,827 | ✘ |

**Why they differ:** pandas filters with `slot_at.between("2025-01-01", "2025-06-30")`. A date-only string becomes **2025-06-30 00:00:00**, so every booking on **30 June after midnight is excluded** (1,032 bookings in 2025 and 1,015 in 2026). SQL uses `slot_at < '2025-07-01'` and includes the full day. **SQL is correct**, because the period is "first half of the year". The fix in pandas:
```python
lo, hi = CONFIG["period_this"]
mask = (bk["slot_at"] >= lo) & (bk["slot_at"] < pd.Timestamp(hi) + pd.Timedelta(days=1))
```
The growth rates happen to be nearly equal (+5.03% vs +5.02%), which is exactly why the bug is easy to miss: **compare levels, not just rates.** The bridge is not affected: `invoice_date` has no time part, so both versions take whole days. Its totals should agree to the rupee. Confirm it by putting SQL query 4's bars beside notebook 05's bridge (₹132,847,274 invoiced in Q2 2026).

**2. Which query would you hand to finance, and could they read it without you?**

**Query 4 (the bridge) together with query 5 (the orphan payments anti-join).** Finance owns the money, and these two answer "where did the invoiced amount go, and which payments can't be matched". They're readable alone because:
- each CTE is a named step (`q2` → `paid` → `classified`) with a comment stating the **grain** ("one row per bar") and the **denominator** ("invoices dated in Q2 2026");
- they use plain SQL (LEFT JOIN, CASE, SUM, GROUP BY) and no Python;
- the cutoff and date range are literal values they can change.

Save a copy as `sql/06_<brief>.sql` with the quarter dates at the top, and link it from the README.

**3. The one index you'd add if this ran weekly, and why not three?**

**`payments (invoice_id)`**, because the weekly finance query joins 470k invoices to 470k payments on `invoice_id`, and that join (plus the anti-join for orphans) is the expensive repeated step. A partial index `WHERE status = 'captured'` makes it smaller still. (If the weekly job were the city-month volume report instead, the choice would be `bookings (city, slot_at)`.)

**Why not three:** every index **slows every load** (the weekly import has to update all of them), takes disk space and memory, and gives the planner more choices. More importantly, each index should be justified by a query you've **measured** with `EXPLAIN ANALYZE` before and after. Add the second one when a second slow, repeated query proves it's needed.